# Chapter 35 : Cosine Similarity

In Chapter 34, you visualized those embeddings using PCA. Now we'll learn how to calculate the similarity between two text embeddings using Python and NumPy.
Our goal is to build cosine similarity from scratch, understand its mathematics, and test it on real sentences using your existing Sentence Transformers model.

What problem are we solving?
Suppose we have these sentences:

Sentence A
"I love machine learning."
Topic: machine learning




Sentence B
"I enjoy learning about artificial intelligence."
Related meaning




Sentence C
"I enjoy playing cricket."
Different topic

Understand the mathematics
Cosine similarity measures the angle between two vectors.
\[
\operatorname{cosine}(A,B)=
\frac{A\cdot B}{\|A\|\|B\|}
\]

There are three components:
- \(A\cdot B\) — the dot product of the two vectors.
- \(\|A\|\) — the magnitude (length) of vector A.
- \(\|B\|\) — the magnitude (length) of vector B.
For example, imagine two vectors pointing in nearly the same direction. Their cosine similarity will be close to 1.

Similarity = 1
Vectors point in the same direction.



Similarity near 0
Vectors are approximately perpendicular.



Similarity = −1
Vectors point in opposite directions.



These are mathematical reference points. A similarity score does not directly translate into a percentage of meaning shared, and the interpretation of a score depends on the embedding model and task.

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def cosine_similarity(vector_a, vector_b):
    dot_product = np.dot(vector_a, vector_b)

    magnitiude_a = np.linalg.norm(vector_a)
    magnitiude_b = np.linalg.norm(vector_b)

    if magnitiude_a == 0 or magnitiude_b == 0:
        raise ValueError("Cannot compare a zero vector")

    similarity = dot_product/(magnitiude_b * magnitiude_b)
    return float(similarity)

In [3]:
# np.linalg.norm(vector_a) = Calculate the length of Vector A



In [4]:
a = np.array([1,0])
b = np.array([1,0])
c = np.array([0,1])
d = np.array([-1,0])


print("Same direction:", cosine_similarity(a, b))
print("Perpendicular:", cosine_similarity(a, c))
print("Opposite direction:", cosine_similarity(a, d))

Same direction: 1.0
Perpendicular: 0.0
Opposite direction: -1.0


In [5]:
# Why do this? It verifies that your function works before you apply it to hundreds of dimensions.

In [8]:
sentence_a = "I love machine learning."
sentence_b = "I enjoy learning about artificial intelligence."
sentence_c = "I enjoy playing cricket."

embeddings = model.encode([sentence_a,sentence_b,sentence_c])

vector_a = embeddings[0]
vector_b = embeddings[1]
vector_c = embeddings[2]

print("Embedding shape:", embeddings.shape)

print(
    "A vs B:",
    round(cosine_similarity(vector_a, vector_b), 4)
)

print(
    "A vs C:",
    round(cosine_similarity(vector_a, vector_c), 4)
)

Embedding shape: (3, 384)
A vs B: 0.6426
A vs C: 0.2147


In [10]:
# Compare multiple sentences automatically 

In [9]:
sentences = [
    "Machine learning is fascinating.",
    "I enjoy studying artificial intelligence.",
    "The cricket team won the match.",
    "Deep learning uses neural networks.",
    "I like playing football."
]

embeddings = model.encode(sentences)

reference_vector = embeddings[0]

for i, sentence in enumerate(sentences):
    score = cosine_similarity(
        reference_vector,
        embeddings[i]
    )

    print(f"{score:.4f} | {sentence}")

1.0000 | Machine learning is fascinating.
0.5275 | I enjoy studying artificial intelligence.
0.0355 | The cricket team won the match.
0.5381 | Deep learning uses neural networks.
0.1226 | I like playing football.


In [13]:
# 5. Why this matters for RAG
#When you build a Retrieval-Augmented Generation (RAG) system, the user asks a question and the system searches for relevant document chunks. Cosine similarity is one way to rank those chunks by their embedding similarity to the question.